# Лабораторна робота №3: метричні методи та машини опорних векторів

**Варіант:** 7 — Titanic (той самий набір, split і ціль `AtRisk`, що в ЛР2).
Предмет дослідження — геометрія алгоритмів: як метрика, масштабування, `k`, ядро, `C` і `γ` змінюють якість і вартість моделі.

In [1]:
from pathlib import Path
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_moons
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, cross_val_predict, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

from src.titanic_workflow import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, RANDOM_STATE, load_titanic, make_cv, make_preprocessor,
    split_features_target, stratified_split,
)

WORK_DIR = Path.cwd().parent
FIGURES = WORK_DIR / "figures"
RESULTS = WORK_DIR / "results"
FIGURES.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)

## 1. Експериментальна задача

**Прикладна мета** (з ЛР2): оцінити ризик загибелі пасажира (`AtRisk = 1`), щоб пріоритезувати допомогу. Важливі обидва класи; FN (пропущений пасажир із ризиком) дорожчі, але вибір порога — тема ПР4, тут порівнюються алгоритми.

**Основна метрика — macro F1.** Позитивний клас є більшістю (61,6%), тож F1 позитивного класу у тривіального baseline дорівнює 0,76 (ЛР2), а macro F1 однаково враховує обидва класи. Додаткові — Accuracy, F1 класу 1, ROC-AUC.

| Гіпотеза | Експеримент | Критерій підтвердження |
|---|---|---|
| H1. Без масштабування kNN і RBF SVM погіршаться, бо `Fare` (0–512) домінує у відстані | фіксовані kNN / linear SVM / RBF SVM × {none, Standard, MinMax} на однакових fold-ах | macro F1 без scaler нижчий щонайменше на 0,02 (≈ 1 std) |
| H2. Для kNN потрібне `k` > 1 і зважування за відстанню, бо в даних є однакові вектори з різними мітками | сітка `k × p × weights` | найкраще `k` ≥ 5; `k = 1` гірше на ≥ 0,02 |
| H3. RBF SVM краща за linear SVM, бо ризик залежить від взаємодій (стать × клас, ЛР2) | linear `C` vs спільний пошук `C × γ` | RBF вища за linear більш ніж на 1 std |
| H4. Неінформативні ознаки зменшать контраст відстаней і якість kNN | додавання 0…256 шумових ознак | медіана `d_min/d_max` зростає, macro F1 падає |

**Інженерні критерії:** затримка прогнозу, розмір моделі в пам'яті, можливість пояснити рішення (сусіди / функція рішення), вартість перенавчання при нових даних.

## 2. План експерименту (зафіксовано до пошуку)

| Елемент | Значення |
|---|---|
| Train / test | стратифікований 80/20, `random_state=42` (712 / 179), як у ЛР2 |
| CV | `StratifiedKFold(5, shuffle=True, random_state=42)` для всіх моделей і пошуків |
| Масштабування | `passthrough`, `StandardScaler`, `MinMaxScaler` (лише числові `Age, SibSp, Parch, Fare` + індикатор пропуску) |
| kNN | `k ∈ {1, 3, 5, 7, 11, 15, 21, 31}`, `p ∈ {1, 2}`, `weights ∈ {uniform, distance}` — 32 конфігурації |
| Linear SVM | `C ∈ {0.01, 0.1, 1, 10, 100}` — 5 |
| RBF SVM | `C ∈ {0.1, 1, 10, 100}` × `γ ∈ {scale, 0.01, 0.1, 1}` — 16 |
| NearestCentroid | без гіперпараметрів — 1 |
| Правило вибору | найбільший CV macro F1; якщо різниця між кандидатами < 1 std, вирішують інженерні критерії (розмір, затримка, пояснюваність) |
| Test | відкривається один раз після вибору |

## 3. Контрольний геометричний експеримент: `make_moons`

520 точок, шум 0,22; друга ознака штучно помножена на **18**. Мітки та взаємне розташування у «змістовному» просторі не змінюються — змінюється лише масштаб.

In [2]:
Xm, ym = make_moons(n_samples=520, noise=0.22, random_state=RANDOM_STATE)
Xm[:, 1] *= 18.0
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.28, stratify=ym, random_state=RANDOM_STATE)
moon_models = {"kNN (k=9)": KNeighborsClassifier(n_neighbors=9), "Linear SVM": SVC(kernel="linear", C=1.0),
               "RBF SVM": SVC(kernel="rbf", C=1.0, gamma="scale")}
xx, yy = np.meshgrid(np.linspace(Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5, 300),
                     np.linspace(Xm[:, 1].min() - 5, Xm[:, 1].max() + 5, 300))
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
moon_rows = []
for col, (name, model) in enumerate(moon_models.items()):
    for row, scaling in enumerate(["none", "standard"]):
        pipe = Pipeline([("scale", "passthrough" if scaling == "none" else StandardScaler()), ("model", clone(model))]).fit(Xm_tr, ym_tr)
        score = f1_score(ym_te, pipe.predict(Xm_te))
        moon_rows.append({"Модель": name, "Масштабування": scaling, "Test F1": score})
        ax = axes[row, col]
        zz = pipe.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
        ax.scatter(Xm_tr[:, 0], Xm_tr[:, 1], c=ym_tr, cmap="coolwarm", s=12, edgecolor="k", lw=0.2)
        if "SVM" in name:
            sv = pipe.named_steps["model"].support_
            ax.scatter(Xm_tr[sv, 0], Xm_tr[sv, 1], s=45, facecolors="none", edgecolors="black", lw=0.8)
        ax.set_title(f"{name}, {'без scaler' if scaling == 'none' else 'StandardScaler'}: F1 = {score:.3f}", fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("x₁")
for ax in axes[:, 0]:
    ax.set_ylabel("x₂ × 18")
fig.suptitle("make_moons: межі рішень до і після стандартизації (кільця — опорні вектори)", y=1.0)
fig.tight_layout()
fig.savefig(FIGURES / "moons_decision_regions.png", dpi=150, bbox_inches="tight")
plt.show()
moons = pd.DataFrame(moon_rows).pivot(index="Модель", columns="Масштабування", values="Test F1")
moons.to_csv(RESULTS / "moons_f1.csv")
display(moons.style.format("{:.3f}"))

Масштабування,none,standard
Модель,,
Linear SVM,0.847,0.847
RBF SVM,0.830,0.951
kNN (k=9),0.857,0.952


## 4. Підготовка даних Titanic

Повторний EDA не потрібен; перевіряємо, що набір і поділ не змінилися з ЛР2.

In [3]:
df = load_titanic()
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = stratified_split(X, y)
print("Об'єктів:", len(X), "| ознак:", X.shape[1], "| класів:", y.nunique(), "| пропусків Age/Embarked:",
      X["Age"].isna().sum(), "/", X["Embarked"].isna().sum())
print("Train / test:", len(X_train), "/", len(X_test), "| P(y=1) train / test:", round(y_train.mean(), 3), "/", round(y_test.mean(), 3))
lr2_ids = pd.read_csv(PROJECT_ROOT / "lab02" / "results" / "test_passenger_ids.csv")["PassengerId"]
print("Test збігається з ЛР2:", set(lr2_ids) == set(X_test.index))
print("Діапазони сирих числових ознак (train):")
display(X_train[NUMERIC_FEATURES].describe().loc[["min", "max", "std"]].style.format("{:.2f}"))

Об'єктів: 891 | ознак: 7 | класів: 2 | пропусків Age/Embarked: 177 / 2
Train / test: 712 / 179 | P(y=1) train / test: 0.617 / 0.615
Test збігається з ЛР2: True
Діапазони сирих числових ознак (train):


,Age,SibSp,Parch,Fare
min,0.42,0.00,0.00,0.00
max,80.00,8.00,6.00,512.33
std,14.71,1.10,0.79,50.55


**Зміст відстані після кодування.** Категоріальні `Pclass`, `Sex`, `Embarked` кодуються one-hot (0/1) і **не масштабуються**: різниця за статтю дає внесок 2 у квадрат евклідової відстані (змінюються два стовпці), за класом каюти — теж 2. Числові ознаки масштабуються. Без масштабування різниця в `Fare` у кілька десятків фунтів повністю «перекриває» стать, хоча саме стать — найсильніша ознака (ЛР2). Після `MinMaxScaler` числова ознака дає внесок не більше 1, тобто категорії мають більшу вагу; після `StandardScaler` — внесок порядку кількох одиниць (і більше для викидів `Fare`, `SibSp`). Отже, вибір scaler фактично задає **відносну вагу категоріальних і числових ознак** у метриці.

In [4]:
cv = make_cv()
scoring = {"macro_f1": "f1_macro", "accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}


def make_pipeline(model, scale="standard", extra_numeric=None):
    if extra_numeric:
        pre = make_preprocessor(scale)
        pre.transformers[0] = (pre.transformers[0][0], pre.transformers[0][1], NUMERIC_FEATURES + extra_numeric)
    else:
        pre = make_preprocessor(scale)
    return Pipeline([("preprocessor", pre), ("model", model)])


def cv_summary(estimator, X_=None, cv_scheme=cv):
    X_ = X_train if X_ is None else X_
    s = cross_validate(estimator, X_, y_train, cv=cv_scheme, scoring=scoring, return_train_score=True)
    return {"CV macro F1": s["test_macro_f1"].mean(), "SD": s["test_macro_f1"].std(ddof=1),
            "Train macro F1": s["train_macro_f1"].mean(), "CV accuracy": s["test_accuracy"].mean(),
            "CV F1 (клас 1)": s["test_f1"].mean(), "CV ROC-AUC": s["test_roc_auc"].mean(), "Fit, с": s["fit_time"].mean()}

## 5. Дослідження масштабування (параметри моделей фіксовані)

In [5]:
fixed_models = {"kNN (k=5, distance, p=2)": KNeighborsClassifier(n_neighbors=5, weights="distance", p=2),
                "Linear SVM (C=1)": SVC(kernel="linear", C=1.0),
                "RBF SVM (C=1, γ=scale)": SVC(kernel="rbf", C=1.0, gamma="scale")}
scale_rows = []
for model_name, model in fixed_models.items():
    for scale, label in [(None, "Без масштабування"), ("standard", "StandardScaler"), ("minmax", "MinMaxScaler")]:
        scale_rows.append({"Модель": model_name, "Перетворення": label, **cv_summary(make_pipeline(clone(model), scale))})
scaling = pd.DataFrame(scale_rows)
scaling.to_csv(RESULTS / "scaling_experiment.csv", index=False)
display(scaling[["Модель", "Перетворення", "CV macro F1", "SD", "CV ROC-AUC", "Fit, с"]].style.format(
    {"CV macro F1": "{:.3f}", "SD": "{:.3f}", "CV ROC-AUC": "{:.3f}", "Fit, с": "{:.3f}"}))

fig, ax = plt.subplots(figsize=(10, 4.5))
pivot = scaling.pivot(index="Модель", columns="Перетворення", values="CV macro F1")[["Без масштабування", "StandardScaler", "MinMaxScaler"]]
pivot_sd = scaling.pivot(index="Модель", columns="Перетворення", values="SD")[pivot.columns]
pivot.plot(kind="bar", yerr=pivot_sd, capsize=3, ax=ax, color=["#c46a5a", "#3f7f93", "#6a8f3f"], rot=0)
ax.set(title="Вплив масштабування на CV macro F1 (5 fold-ів, ± SD)", ylabel="CV macro F1", xlabel="", ylim=(0.5, 0.9))
ax.legend(title="", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "scaling_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

,Модель,Перетворення,CV macro F1,SD,CV ROC-AUC,"Fit, с"
0,"kNN (k=5, distance, p=2)",Без масштабування,0.676,0.029,0.738,0.012
1,"kNN (k=5, distance, p=2)",StandardScaler,0.753,0.022,0.819,0.011
2,"kNN (k=5, distance, p=2)",MinMaxScaler,0.752,0.019,0.817,0.010
3,Linear SVM (C=1),Без масштабування,0.768,0.004,0.833,1.974
4,Linear SVM (C=1),StandardScaler,0.768,0.004,0.816,0.018
5,Linear SVM (C=1),MinMaxScaler,0.768,0.004,0.811,0.016
6,"RBF SVM (C=1, γ=scale)",Без масштабування,0.589,0.023,0.746,0.018
7,"RBF SVM (C=1, γ=scale)",StandardScaler,0.791,0.012,0.843,0.017
8,"RBF SVM (C=1, γ=scale)",MinMaxScaler,0.784,0.017,0.837,0.018


## 6. Метричні класифікатори: NearestCentroid і kNN

In [6]:
SCALE_KNN = scaling[scaling["Модель"].str.startswith("kNN")].sort_values("CV macro F1", ascending=False).iloc[0]["Перетворення"]
SCALE_MAP = {"Без масштабування": None, "StandardScaler": "standard", "MinMaxScaler": "minmax"}
print("Scaler для kNN і NearestCentroid (за п.5):", SCALE_KNN)

centroid_summary = cv_summary(make_pipeline(NearestCentroid(), SCALE_MAP[SCALE_KNN]))
print("NearestCentroid:", {k: round(v, 3) for k, v in centroid_summary.items()})

knn_search = GridSearchCV(make_pipeline(KNeighborsClassifier(), SCALE_MAP[SCALE_KNN]),
                          {"model__n_neighbors": [1, 3, 5, 7, 11, 15, 21, 31], "model__p": [1, 2],
                           "model__weights": ["uniform", "distance"]},
                          scoring=scoring, refit="macro_f1", cv=cv, return_train_score=True, n_jobs=2).fit(X_train, y_train)
knn_cv = pd.DataFrame(knn_search.cv_results_)
knn_table = knn_cv[["param_model__n_neighbors", "param_model__p", "param_model__weights", "mean_test_macro_f1", "std_test_macro_f1",
                    "mean_train_macro_f1", "mean_test_roc_auc"]].rename(columns=lambda c: c.replace("param_model__", "")).sort_values(
    "mean_test_macro_f1", ascending=False)
knn_table.to_csv(RESULTS / "knn_grid.csv", index=False)
display(knn_table.head(8).style.format({"mean_test_macro_f1": "{:.3f}", "std_test_macro_f1": "{:.3f}",
                                        "mean_train_macro_f1": "{:.3f}", "mean_test_roc_auc": "{:.3f}"}))
print("Найкращий kNN:", knn_search.best_params_, f"CV macro F1 = {knn_search.best_score_:.3f}")

Scaler для kNN і NearestCentroid (за п.5): StandardScaler


NearestCentroid: {'CV macro F1': np.float64(0.743), 'SD': np.float64(0.048), 'Train macro F1': np.float64(0.753), 'CV accuracy': np.float64(0.758), 'CV F1 (клас 1)': np.float64(0.806), 'CV ROC-AUC': np.float64(0.844), 'Fit, с': np.float64(0.011)}


,n_neighbors,p,weights,mean_test_macro_f1,std_test_macro_f1,mean_train_macro_f1,mean_test_roc_auc
16,11,1,uniform,0.789,0.032,0.822,0.854
20,15,1,uniform,0.785,0.035,0.812,0.855
24,21,1,uniform,0.781,0.026,0.802,0.854
12,7,1,uniform,0.779,0.029,0.832,0.848
22,15,2,uniform,0.779,0.025,0.803,0.847
10,5,2,uniform,0.776,0.017,0.838,0.836
8,5,1,uniform,0.776,0.018,0.844,0.841
18,11,2,uniform,0.773,0.031,0.812,0.847


Найкращий kNN: {'model__n_neighbors': 11, 'model__p': 1, 'model__weights': 'uniform'} CV macro F1 = 0.789


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, weights in zip(axes, ["uniform", "distance"]):
    for p_value, color in [(1, "#c46a5a"), (2, "#3f7f93")]:
        part = knn_table[(knn_table["weights"] == weights) & (knn_table["p"] == p_value)].sort_values("n_neighbors")
        ax.errorbar(part["n_neighbors"], part["mean_test_macro_f1"], yerr=part["std_test_macro_f1"], marker="o",
                    capsize=3, color=color, label=f"CV, p={p_value} ({'мангеттенська' if p_value == 1 else 'евклідова'})")
        ax.plot(part["n_neighbors"], part["mean_train_macro_f1"], ls="--", color=color, alpha=0.6, label=f"train, p={p_value}")
    ax.set(title=f"kNN, weights = {weights}", xlabel="k (кількість сусідів)", ylabel="macro F1", xscale="log",
           xticks=[1, 3, 5, 7, 11, 15, 21, 31])
    ax.set_xticklabels([1, 3, 5, 7, 11, 15, 21, 31])
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "knn_k_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Машини опорних векторів

In [8]:
SCALE_SVM = scaling[scaling["Модель"].str.startswith("RBF")].sort_values("CV macro F1", ascending=False).iloc[0]["Перетворення"]
print("Scaler для SVM (за п.5, RBF):", SCALE_SVM)
linear_search = GridSearchCV(make_pipeline(SVC(kernel="linear"), SCALE_MAP[SCALE_SVM]), {"model__C": [0.01, 0.1, 1, 10, 100]},
                             scoring=scoring, refit="macro_f1", cv=cv, return_train_score=True, n_jobs=2).fit(X_train, y_train)
rbf_search = GridSearchCV(make_pipeline(SVC(kernel="rbf"), SCALE_MAP[SCALE_SVM]),
                          {"model__C": [0.1, 1, 10, 100], "model__gamma": ["scale", 0.01, 0.1, 1]},
                          scoring=scoring, refit="macro_f1", cv=cv, return_train_score=True, n_jobs=2).fit(X_train, y_train)

svm_rows = []
for kernel, search in [("linear", linear_search), ("rbf", rbf_search)]:
    res = pd.DataFrame(search.cv_results_)
    for i, params in enumerate(res["params"]):
        n_sv = make_pipeline(SVC(kernel=kernel, **{k.split("__")[1]: v for k, v in params.items()}),
                             SCALE_MAP[SCALE_SVM]).fit(X_train, y_train).named_steps["model"].n_support_.sum()
        svm_rows.append({"kernel": kernel, "C": params["model__C"], "gamma": params.get("model__gamma", "—"),
                         "CV macro F1": res.loc[i, "mean_test_macro_f1"], "SD": res.loc[i, "std_test_macro_f1"],
                         "Train macro F1": res.loc[i, "mean_train_macro_f1"], "CV ROC-AUC": res.loc[i, "mean_test_roc_auc"],
                         "Опорних векторів": n_sv})
svm_table = pd.DataFrame(svm_rows)
svm_table.to_csv(RESULTS / "svm_grid.csv", index=False)
display(svm_table.style.format({"CV macro F1": "{:.3f}", "SD": "{:.3f}", "Train macro F1": "{:.3f}", "CV ROC-AUC": "{:.3f}"}))
print("Найкращий linear SVM:", linear_search.best_params_, f"{linear_search.best_score_:.3f}")
print("Найкращий RBF SVM:", rbf_search.best_params_, f"{rbf_search.best_score_:.3f}")

Scaler для SVM (за п.5, RBF): StandardScaler


,kernel,C,gamma,CV macro F1,SD,Train macro F1,CV ROC-AUC,Опорних векторів
0,linear,0.010000,—,0.767,0.007,0.769,0.853,467
1,linear,0.100000,—,0.768,0.003,0.768,0.825,347
2,linear,1.000000,—,0.768,0.003,0.768,0.816,331
3,linear,10.000000,—,0.768,0.003,0.769,0.826,334
4,linear,100.000000,—,0.768,0.003,0.769,0.814,351
5,rbf,0.100000,scale,0.760,0.029,0.786,0.846,491
6,rbf,0.100000,0.010000,0.408,0.025,0.414,0.848,548
7,rbf,0.100000,0.100000,0.763,0.030,0.779,0.851,491
8,rbf,0.100000,1,0.512,0.028,0.548,0.835,557
9,rbf,1.000000,scale,0.791,0.011,0.830,0.843,348


Найкращий linear SVM: {'model__C': 0.1} 0.768
Найкращий RBF SVM: {'model__C': 100, 'model__gamma': 0.01} 0.800


In [9]:
rbf_part = svm_table[svm_table["kernel"] == "rbf"].copy()
rbf_part["gamma"] = rbf_part["gamma"].astype(str)
order = ["scale", "0.01", "0.1", "1"]
heat = rbf_part.pivot(index="C", columns="gamma", values="CV macro F1")[order]
heat_train = rbf_part.pivot(index="C", columns="gamma", values="Train macro F1")[order]
heat_sv = rbf_part.pivot(index="C", columns="gamma", values="Опорних векторів")[order]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="viridis", ax=axes[0], cbar_kws={"label": "CV macro F1"})
axes[0].set(title="RBF SVM: CV macro F1", xlabel="γ", ylabel="C")
sns.heatmap(heat_train - heat, annot=True, fmt=".3f", cmap="Reds", ax=axes[1], cbar_kws={"label": "train − CV"})
axes[1].set(title="Розрив train − CV", xlabel="γ", ylabel="C")
sns.heatmap(heat_sv, annot=True, fmt="d", cmap="Blues", ax=axes[2], cbar_kws={"label": "опорних векторів"})
axes[2].set(title="Кількість опорних векторів (fit на train)", xlabel="γ", ylabel="C")
fig.tight_layout()
fig.savefig(FIGURES / "rbf_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

`γ = scale` у scikit-learn дорівнює $1 / (m \cdot \mathrm{Var}(X))$, де $m$ — кількість ознак після preprocessing; для цього простору воно близьке до 0,14 (обчислено нижче), тобто між сітковими 0,1 і 1.

In [10]:
Xt = make_preprocessor(SCALE_MAP[SCALE_SVM]).fit_transform(X_train)
print(f"γ(scale) = {1 / (Xt.shape[1] * Xt.var()):.4f} для {Xt.shape[1]} ознак")

# Чому linear SVM майже не залежить від C: порівнюємо її OOF-прогнози з правилом «чоловік → не вижив»
lin_oof = cross_val_predict(linear_search.best_estimator_, X_train, y_train, cv=cv)
sex_rule = (X_train["Sex"] == "male").astype(int).to_numpy()
print(f"Linear SVM збігається з правилом статі для {np.mean(lin_oof == sex_rule):.1%} пасажирів; "
      f"macro F1 правила статі = {f1_score(y_train, sex_rule, average='macro'):.3f}")

γ(scale) = 0.1368 для 13 ознак
Linear SVM збігається з правилом статі для 100.0% пасажирів; macro F1 правила статі = 0.768


## 8. Контрольований експеримент із розмірністю

До числових ознак train додаються незалежні шумові стовпці `U(0, 1)` із фіксованим seed. Вони проходять той самий scaler, що й справжні числові ознаки, тому мають порівнянний діапазон. Для кожної кількості шумових ознак: медіана $d_{min}/d_{max}$ на підмножині 300 об'єктів і CV macro F1 **зафіксованого** найкращого kNN. Після експерименту штучні ознаки відкидаються.

In [11]:
rng = np.random.default_rng(RANDOM_STATE)
best_knn_params = {k.split("__")[1]: v for k, v in knn_search.best_params_.items()}
noise_rows = []
for n_noise in [0, 8, 32, 128, 256]:
    noise_cols = [f"noise_{i}" for i in range(n_noise)]
    X_aug = X_train.copy()
    if n_noise:
        X_aug = pd.concat([X_aug, pd.DataFrame(rng.uniform(0, 1, (len(X_train), n_noise)), index=X_train.index, columns=noise_cols)], axis=1)
    pipe = make_pipeline(KNeighborsClassifier(**best_knn_params), SCALE_MAP[SCALE_KNN], extra_numeric=noise_cols or None)
    Z = pipe.named_steps["preprocessor"].fit_transform(X_aug)
    sub = Z[:300]
    dist = np.linalg.norm(sub[:, None, :] - sub[None, :, :], ord=best_knn_params["p"], axis=2)
    np.fill_diagonal(dist, np.nan)
    ratio = np.nanmin(dist, axis=1) / np.nanmax(dist, axis=1)
    summary = cv_summary(pipe, X_aug)
    noise_rows.append({"Шумових ознак": n_noise, "Усього ознак": Z.shape[1], "Медіана d_min/d_max": np.median(ratio),
                       "kNN CV macro F1": summary["CV macro F1"], "SD": summary["SD"]})
noise = pd.DataFrame(noise_rows)
noise.to_csv(RESULTS / "dimensionality_experiment.csv", index=False)
display(noise.style.format({"Медіана d_min/d_max": "{:.3f}", "kNN CV macro F1": "{:.3f}", "SD": "{:.3f}"}))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(noise["Шумових ознак"], noise["Медіана d_min/d_max"], marker="o", color="#c46a5a", label="медіана d_min/d_max")
ax.set(xscale="symlog", xlabel="Кількість шумових ознак (symlog)", ylabel="d_min / d_max", title="Концентрація відстаней і якість kNN")
ax2 = ax.twinx()
ax2.errorbar(noise["Шумових ознак"], noise["kNN CV macro F1"], yerr=noise["SD"], marker="s", capsize=3, color="#3f7f93", label="kNN CV macro F1")
ax2.set_ylabel("CV macro F1")
ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="center right", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES / "dimensionality.png", dpi=160, bbox_inches="tight")
plt.show()

,Шумових ознак,Усього ознак,Медіана d_min/d_max,kNN CV macro F1,SD
0,0,13,0.017,0.789,0.036
1,8,21,0.241,0.757,0.043
2,32,45,0.493,0.676,0.038
3,128,141,0.710,0.582,0.063
4,256,269,0.788,0.537,0.039


## 9. Порівняння моделей

Логістична регресія з ЛР2 включена як контрольна точка без повторного підбору. Час прогнозування — медіана 7 запусків (після прогріву) на фіксованих 1 000 рядках (вибірка з поверненням із train); розмір — довжина `pickle` всього pipeline.

In [12]:
best_models = {
    "NearestCentroid": make_pipeline(NearestCentroid(), SCALE_MAP[SCALE_KNN]),
    "kNN": knn_search.best_estimator_,
    "Linear SVM": linear_search.best_estimator_,
    "RBF SVM": rbf_search.best_estimator_,
    "Logistic Regression (ЛР2)": make_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), "standard"),
}
params_text = {"NearestCentroid": f"scaler={SCALE_KNN}", "kNN": str(best_knn_params),
               "Linear SVM": f"C={linear_search.best_params_['model__C']}",
               "RBF SVM": f"C={rbf_search.best_params_['model__C']}, γ={rbf_search.best_params_['model__gamma']}",
               "Logistic Regression (ЛР2)": "C=1"}
X_bench = X_train.sample(1000, replace=True, random_state=RANDOM_STATE)
compare_rows = []
for name, est in best_models.items():
    summary = cv_summary(clone(est))
    fitted = clone(est)
    fit_times = []
    for _ in range(3):
        start = time.perf_counter(); fitted = clone(est).fit(X_train, y_train); fit_times.append(time.perf_counter() - start)
    fitted.predict(X_bench)
    pred_times = []
    for _ in range(7):
        start = time.perf_counter(); fitted.predict(X_bench); pred_times.append(time.perf_counter() - start)
    model = fitted.named_steps["model"]
    if name == "NearestCentroid":
        stored = len(model.centroids_)
    elif name == "kNN":
        stored = model.n_samples_fit_
    else:
        stored = int(model.n_support_.sum()) if hasattr(model, "n_support_") else 0
    compare_rows.append({"Модель": name, "Гіперпараметри": params_text[name], "CV macro F1": summary["CV macro F1"],
                         "SD": summary["SD"], "CV ROC-AUC": summary["CV ROC-AUC"], "Fit, мс": np.median(fit_times) * 1000,
                         "Predict 1000, мс": np.median(pred_times) * 1000, "Розмір, КБ": len(pickle.dumps(fitted)) / 1024,
                         "Збережено об'єктів": stored})
comparison = pd.DataFrame(compare_rows).sort_values("CV macro F1", ascending=False).reset_index(drop=True)
comparison.to_csv(RESULTS / "model_comparison.csv", index=False)
display(comparison.style.format({"CV macro F1": "{:.3f}", "SD": "{:.3f}", "CV ROC-AUC": "{:.3f}", "Fit, мс": "{:.1f}",
                                 "Predict 1000, мс": "{:.2f}", "Розмір, КБ": "{:.1f}"}))

,Модель,Гіперпараметри,CV macro F1,SD,CV ROC-AUC,"Fit, мс","Predict 1000, мс","Розмір, КБ",Збережено об'єктів
0,RBF SVM,"C=100, γ=0.01",0.800,0.030,0.847,25.3,15.69,39.6,296
1,kNN,"{'n_neighbors': 11, 'p': 1, 'weights': 'uniform'}",0.789,0.036,0.854,9.4,15.89,94.4,712
2,Logistic Regression (ЛР2),C=1,0.789,0.018,0.854,12.4,4.32,3.4,0
3,Linear SVM,C=0.1,0.768,0.004,0.825,15.8,10.77,45.8,347
4,NearestCentroid,scaler=StandardScaler,0.743,0.048,0.844,10.5,5.70,3.7,2


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = dict(zip(comparison["Модель"], ["#3f7f93", "#c46a5a", "#6a8f3f", "#8a6aa8", "#d49a3a"]))
for _, r in comparison.iterrows():
    axes[0].errorbar(r["Predict 1000, мс"], r["CV macro F1"], yerr=r["SD"], marker="o", ms=9, capsize=4, color=colors[r["Модель"]], label=r["Модель"])
    axes[1].errorbar(r["Розмір, КБ"], r["CV macro F1"], yerr=r["SD"], marker="o", ms=9, capsize=4, color=colors[r["Модель"]])
    axes[1].annotate(f"{r['Збережено об\'єктів']} об'єктів", (r["Розмір, КБ"], r["CV macro F1"]), textcoords="offset points",
                     xytext=(6, -12), fontsize=8)
axes[0].set(xscale="log", title="Якість vs затримка", xlabel="Час прогнозу 1000 об'єктів, мс (log)", ylabel="CV macro F1 ± SD")
axes[1].set(xscale="log", title="Якість vs розмір моделі", xlabel="Розмір pickle, КБ (log)", ylabel="CV macro F1 ± SD")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "quality_cost_tradeoff.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Аналіз розбіжностей kNN та SVM (out-of-fold)

Кожен об'єкт train прогнозується моделями, навченими на інших fold-ах. Для об'єктів, де kNN і найкраща SVM не погоджуються, у тому самому fold-і знаходимо `k` сусідів (їхні класи й відстані) і значення `decision_function` SVM.

In [14]:
best_svm_name = "RBF SVM" if rbf_search.best_score_ >= linear_search.best_score_ else "Linear SVM"
best_svm = best_models[best_svm_name]
knn_est = best_models["kNN"]
records = []
for fold, (tr, va) in enumerate(cv.split(X_train, y_train), 1):
    knn_fold = clone(knn_est).fit(X_train.iloc[tr], y_train.iloc[tr])
    svm_fold = clone(best_svm).fit(X_train.iloc[tr], y_train.iloc[tr])
    Xva = X_train.iloc[va]
    knn_pred, svm_pred = knn_fold.predict(Xva), svm_fold.predict(Xva)
    decision = svm_fold.decision_function(Xva)
    Z_tr = knn_fold.named_steps["preprocessor"].transform(X_train.iloc[tr])
    Z_va = knn_fold.named_steps["preprocessor"].transform(Xva)
    dist, idx = knn_fold.named_steps["model"].kneighbors(Z_va)
    y_tr = y_train.iloc[tr].to_numpy()
    for j, pid in enumerate(Xva.index):
        neigh = y_tr[idx[j]]
        records.append({"PassengerId": pid, "fold": fold, "y_true": y_train.loc[pid], "kNN": knn_pred[j], "SVM": svm_pred[j],
                        "SVM decision": decision[j], "Сусіди клас 1": f"{neigh.sum()}/{len(neigh)}",
                        "Відстані сусідів": ", ".join(f"{d:.2f}" for d in dist[j]),
                        "Однакових сусідів (d=0)": int((dist[j] < 1e-9).sum())})
oof = pd.DataFrame(records).set_index("PassengerId").loc[X_train.index]
oof.to_csv(RESULTS / "oof_predictions.csv")
dis = oof[oof["kNN"] != oof["SVM"]].join(X_train)
dis["Правий"] = np.where(dis["kNN"] == dis["y_true"], "kNN", "SVM")
dis.to_csv(RESULTS / "disagreements.csv")
print(f"OOF macro F1: kNN = {f1_score(y_train, oof['kNN'], average='macro'):.3f}, {best_svm_name} = {f1_score(y_train, oof['SVM'], average='macro'):.3f}")
print("Розбіжностей:", len(dis), "з", len(oof), "| правий kNN:", (dis["Правий"] == "kNN").sum(), "| правий SVM:", (dis["Правий"] == "SVM").sum())
both_wrong = oof[(oof["kNN"] != oof["y_true"]) & (oof["SVM"] != oof["y_true"])]
print("Обидві моделі помилились (і погодились між собою):", len(both_wrong))
print("Розбіжності за статтю і класом:")
display(dis.groupby(["Sex", "Pclass", "Правий"]).size().unstack(fill_value=0))

OOF macro F1: kNN = 0.789, RBF SVM = 0.801
Розбіжностей: 61 з 712 | правий kNN: 26 | правий SVM: 35
Обидві моделі помилились (і погодились між собою): 102
Розбіжності за статтю і класом:


Правий         SVM  kNN
Sex    Pclass          
female 1         1    0
       2         0    1
       3        12   14
male   1        21   11
       3         1    0

In [15]:
cols = ["y_true", "kNN", "SVM", "Правий", "SVM decision", "Сусіди клас 1", "Відстані сусідів", "Однакових сусідів (d=0)",
        "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
detail = pd.concat([dis[dis["Правий"] == "kNN"].sort_values("SVM decision", key=abs).head(6),
                    dis[dis["Правий"] == "SVM"].sort_values("SVM decision", key=abs, ascending=False).head(6)])
display(detail[cols].style.format({"SVM decision": "{:+.2f}", "Age": "{:.0f}", "Fare": "{:.1f}"}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(data=oof, x="SVM decision", hue=(oof["kNN"] != oof["SVM"]).map({True: "розбіжність", False: "згода"}),
             bins=40, stat="density", common_norm=False, ax=axes[0], palette={"розбіжність": "#c46a5a", "згода": "#3f7f93"})
axes[0].get_legend().set_title("")
axes[0].axvline(0, color="black", ls="--")
axes[0].set(title="Функція рішення SVM: згода vs розбіжність kNN/SVM", xlabel="decision_function (OOF)", ylabel="щільність")
counts = dis.groupby(["Правий"]).size()
dis["група"] = dis["Sex"].map({"male": "чол.", "female": "жін."}) + ", " + dis["Pclass"].astype(str) + " кл."
sns.countplot(data=dis, x="група", hue="Правий", ax=axes[1], palette={"kNN": "#6a8f3f", "SVM": "#8a6aa8"},
              order=sorted(dis["група"].unique()))
axes[1].set(title="Розбіжності kNN/SVM за групами пасажирів", xlabel="", ylabel="кількість")
fig.tight_layout()
fig.savefig(FIGURES / "disagreements.png", dpi=160, bbox_inches="tight")
plt.show()

,y_true,kNN,SVM,Правий,SVM decision,Сусіди клас 1,Відстані сусідів,Однакових сусідів (d=0),Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,,,,,,,,,
371,0,0,1,kNN,+0.02,3/11,"0.80, 1.54, 1.91, 2.00, 2.09, 2.19, 2.20, 2.31, 2.48, 2.55, 2.59",0,1,male,25,1,0,55.4,C
504,1,1,0,kNN,-0.08,6/11,"0.48, 0.50, 0.56, 0.60, 0.65, 0.80, 0.88, 0.96, 1.02, 1.11, 1.12",0,3,female,37,0,0,9.6,S
475,1,1,0,kNN,-0.09,6/11,"0.05, 0.11, 0.12, 0.17, 0.27, 0.35, 0.35, 0.35, 0.42, 0.64, 0.66",0,3,female,22,0,0,9.8,S
643,1,1,0,kNN,-0.12,6/11,"0.52, 1.17, 1.53, 1.62, 1.85, 2.41, 2.60, 2.90, 3.39, 3.47, 3.98",0,3,female,2,3,2,27.9,S
294,1,1,0,kNN,-0.17,6/11,"0.09, 0.10, 0.10, 0.17, 0.17, 0.18, 0.18, 0.24, 0.25, 0.32, 0.47",0,3,female,24,0,0,8.8,S
535,1,1,0,kNN,-0.24,6/11,"0.07, 0.08, 0.09, 0.16, 0.24, 0.32, 0.39, 0.54, 0.54, 0.55, 0.62",0,3,female,30,0,0,8.7,S
558,1,0,1,SVM,+1.40,3/11,"3.76, 3.96, 4.00, 4.34, 4.56, 4.58, 5.23, 5.54, 5.72, 5.85, 5.85",0,1,male,nan,0,0,227.5,C
12,0,1,0,SVM,-1.07,6/11,"0.71, 0.79, 2.15, 2.15, 2.31, 2.31, 2.34, 2.40, 2.43, 2.54, 2.54",0,1,female,58,0,0,26.6,S
537,1,0,1,SVM,+1.02,5/11,"0.00, 0.17, 0.19, 0.22, 0.23, 0.30, 0.45, 0.61, 0.68, 0.68, 0.82",1,1,male,45,0,0,26.6,S


## 11–12. Вибір фінальної моделі та одноразове оцінювання на test

Обґрунтування вибору наведено в `README.md`; правило зафіксоване в п.2. Нижче фінальна модель навчається на всьому train і один раз оцінюється на test.

In [16]:
ranked = comparison[comparison["Модель"] != "Logistic Regression (ЛР2)"].reset_index(drop=True)
leader = ranked.iloc[0]
close = ranked[leader["CV macro F1"] - ranked["CV macro F1"] < leader["SD"]]
print("Лідер за CV:", leader["Модель"], f"{leader['CV macro F1']:.3f} ± {leader['SD']:.3f}")
print("У межах 1 SD від лідера:", list(close["Модель"]))
FINAL_NAME = close.sort_values("Розмір, КБ").iloc[0]["Модель"] if len(close) > 1 else leader["Модель"]
print("Фінальна модель (найменша серед рівноцінних):", FINAL_NAME)

final = clone(best_models[FINAL_NAME]).fit(X_train, y_train)
y_pred = final.predict(X_test)
score = final.decision_function(X_test) if hasattr(final, "decision_function") else final.predict_proba(X_test)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
final_metrics = pd.DataFrame([{"Модель": FINAL_NAME, "Accuracy": accuracy_score(y_test, y_pred),
                               "Precision": precision_score(y_test, y_pred), "Recall": recall_score(y_test, y_pred),
                               "F1 (клас 1)": f1_score(y_test, y_pred), "Macro F1": f1_score(y_test, y_pred, average="macro"),
                               "ROC-AUC": roc_auc_score(y_test, score), "TN": tn, "FP": fp, "FN": fn, "TP": tp}])
final_metrics.to_csv(RESULTS / "test_metrics.csv", index=False)
display(final_metrics.style.format({c: "{:.3f}" for c in ["Accuracy", "Precision", "Recall", "F1 (клас 1)", "Macro F1", "ROC-AUC"]}))
cv_value = comparison.set_index("Модель").loc[FINAL_NAME]
print(f"CV macro F1 = {cv_value['CV macro F1']:.3f} ± {cv_value['SD']:.3f}; test macro F1 = {final_metrics.loc[0, 'Macro F1']:.3f}")

Лідер за CV: RBF SVM 0.800 ± 0.030
У межах 1 SD від лідера: ['RBF SVM', 'kNN']
Фінальна модель (найменша серед рівноцінних): RBF SVM


,Модель,Accuracy,Precision,Recall,F1 (клас 1),Macro F1,ROC-AUC,TN,FP,FN,TP
0,RBF SVM,0.827,0.816,0.927,0.868,0.808,0.820,46,23,8,102


CV macro F1 = 0.800 ± 0.030; test macro F1 = 0.808
